In [1]:
import gc
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,TensorDataset
from rating_dataset import RatingDataset
from model import SparseLandauScoreField
from utils import preprocess_face_dataset
ROOT_DIR=Path(r'D:\MyJupyter\Works\FaceScore')
if str(ROOT_DIR) not in __import__('sys').path:
    __import__('sys').path.insert(0,str(ROOT_DIR))
reference_notebook=json.loads((ROOT_DIR/'train_compare.ipynb').read_text(encoding='utf-8'))
for cell_index in (2,4,7,9):
    exec(''.join(reference_notebook['cells'][cell_index]['source']),globals())
cell12_source=''.join(reference_notebook['cells'][12]['source'])
exec(cell12_source.split('optimizer=torch.optim.AdamW',1)[0],globals())
SEEDS=[42,123,2026,3407,777]
TEST_RAW_DIR=ROOT_DIR/'data'/'test_data'
TEST_DIR=ROOT_DIR/'data'/'test_face_data'
TEST_MASK_DIR=ROOT_DIR/'data'/'test_face_masks'
TABLE_DIR=ROOT_DIR/'outputs'/'tables'
TABLE_DIR.mkdir(parents=True,exist_ok=True)
EPOCHS=200
PATIENCE=40
NLL_PATIENCE=6
MIN_DELTA=1e-5
if not TEST_RAW_DIR.is_dir():
    raise FileNotFoundError(f'找不到原始测试目录: {TEST_RAW_DIR}')
test_images=sorted(path for path in TEST_RAW_DIR.iterdir() if path.suffix.lower() in {'.jpg','.jpeg','.png'})
missing_test=[path for path in test_images if not (TEST_DIR/path.name).is_file() or not (TEST_MASK_DIR/f'{path.stem}.png').is_file() or not (TEST_DIR/f'{path.stem}.txt').is_file()]
if missing_test:
    preprocess_face_dataset(src_dir=str(TEST_RAW_DIR),dst_dir=str(TEST_DIR),mask_dir=str(TEST_MASK_DIR),size=448,expand=1.75)
test_dataset=RatingDataset(str(TEST_DIR),mask_dir=str(TEST_MASK_DIR),augment=False,repeat=1,image_size=336,soft_target=False)
test_loader=DataLoader(test_dataset,batch_size=OURS_FEATURE_BATCH,shuffle=False,num_workers=0,pin_memory=USE_AMP)
@torch.inference_mode()
def extract_test_features():
    feature_parts=[]
    target_parts=[]
    student.eval()
    for inputs,targets in test_loader:
        inputs=inputs.to(device,non_blocking=True)
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=USE_AMP):
            features=student.encode_features(inputs)
        feature_parts.append(features.float().cpu())
        target_parts.append(targets.float().cpu())
    return torch.cat(feature_parts),torch.cat(target_parts)
test_feature_values,test_targets=extract_test_features()
class ParameterMatchedMLP(SparseLandauScoreField):
    def __init__(self,backbone):
        super().__init__(pretrained=False,backbone=backbone,dropout=0.10,train_last_stage=False,width=24,prior_mix=0.0,use_entropy=False,use_high_resolution=False,scales=(224,),grid_side=16,use_evidence_head=True,evidence_base=BASE_CONCENTRATION,use_global_token=True)
        del self.feature_projection
        del self.direct_residual_head
        del self.force_head
        del self.temperature_head
        del self.free_energy_head
        self.mlp_trunk=nn.Sequential(nn.LayerNorm(self.summary_dim),nn.Linear(self.summary_dim,86),nn.GELU(),nn.Dropout(0.10),nn.Linear(86,46),nn.GELU(),nn.Dropout(0.10))
        self.logits_head=nn.Linear(46,10)
        self.temperature_head=nn.Linear(46,1)
        nn.init.zeros_(self.temperature_head.weight)
        nn.init.constant_(self.temperature_head.bias,-1.2)
    def _predict_from_summary(self,summary):
        latent=self.mlp_trunk(summary.float())
        temperature=0.28+0.42*torch.sigmoid(self.temperature_head(latent).float())
        logits=self.logits_head(latent).float()
        log_probs=F.log_softmax(logits/self.output_temperature,dim=1)
        evidence=(1.5/temperature).clamp(0.05,20.0)
        alpha=self.evidence_base*self.prior_probs.to(device=summary.device,dtype=summary.dtype).view(1,10)+evidence*log_probs.exp()
        return log_probs,alpha
PARAMETER_COUNTS={'Full SSE':72185,'Parameter-matched MLP':72189}
def reset_mlp_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
def make_mlp_optimizer(net):
    decay=[]
    no_decay=[]
    for name,param in net.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim<2 or 'bias' in name.lower() or 'norm' in name.lower():
            no_decay.append(param)
        else:
            decay.append(param)
    return torch.optim.AdamW([{'params':decay,'lr':LEARNING_RATE*1.3,'weight_decay':WEIGHT_DECAY*0.5},{'params':no_decay,'lr':LEARNING_RATE*1.3,'weight_decay':0.0}],lr=LEARNING_RATE*1.3)
def mlp_schedule_lambda(epoch):
    if epoch<WARMUP_EPOCHS:
        return 0.3+0.7*(epoch+1)/WARMUP_EPOCHS
    schedule_horizon=min(EPOCHS,60)
    progress=min(1.0,(epoch-WARMUP_EPOCHS+1)/max(1,schedule_horizon-WARMUP_EPOCHS))
    return (1e-4/LEARNING_RATE)+(1.0-1e-4/LEARNING_RATE)*0.5*(1.0+np.cos(np.pi*progress))
def predict_feature_distribution(net,features):
    net.eval()
    parts=[]
    with torch.inference_mode():
        for start in range(0,len(features),128):
            batch=features[start:start+128].to(device,non_blocking=True)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=USE_AMP):
                output,_=net.forward_from_features(batch,return_pair=False)
            parts.append(output.float().exp().cpu())
    return torch.cat(parts)
if len(train_feature_values)!=2335 or len(val_feature_values)!=260 or len(test_feature_values)!=1480:
    raise RuntimeError(f'固定划分数量错误: train={len(train_feature_values)}, val={len(val_feature_values)}, test={len(test_feature_values)}')
feature_train_dataset=TensorDataset(train_feature_values,train_targets)
feature_val_dataset=TensorDataset(val_feature_values,val_targets)
train_loader=DataLoader(CountDataset(feature_train_dataset,train_counts,train_targets),batch_size=128,shuffle=True,num_workers=0,pin_memory=USE_AMP)
val_loader=DataLoader(feature_val_dataset,batch_size=128,shuffle=False,num_workers=0,pin_memory=USE_AMP)
loss_fn=DistributionLoss(prior=prior,alpha=BASE_CONCENTRATION,dm_weight=0.02,ce_weight=0.50,emd_weight=1.00,mean_weight=1.00,rank_weight=0.20,rank_temperature=0.30,rdrop_weight=0.02,tail_weight=0.05,tail_margin=0.02,target_smoothing=0.35,entropy_weight=0.02,count_weight_power=0.25,count_weight_max=1.5,dm_per_rating=True,spread_weight=0.05,field_smooth_weight=0.002).to(device)
def train_mlp_seed(seed):
    reset_mlp_seed(seed)
    net=ParameterMatchedMLP(tip_adapter).to(device)
    net.set_feature_stats(train_feature_values.mean(dim=0),train_feature_values.std(dim=0))
    net.set_prior(prior)
    optimizer=make_mlp_optimizer(net)
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=mlp_schedule_lambda)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP) if device.type=='cuda' else None
    ema=TrainableEMA(net,decay=EMA_DECAY)
    best_score=float('inf')
    best_nll=float('inf')
    best_epoch=0
    best_val=None
    best_state=None
    score_wait=0
    nll_wait=0
    for epoch in range(1,EPOCHS+1):
        train_result=train_epoch(net,optimizer,scaler,ema)
        ema.apply(net)
        try:
            val_result=evaluate_epoch(net)
            score=select_score(val_result)
            improved=np.isfinite(score) and score<best_score-MIN_DELTA
            if improved:
                best_score=score
                best_epoch=epoch
                best_val=dict(val_result)
                best_state={name:tensor.detach().cpu().clone() for name,tensor in net.state_dict().items()}
            score_wait=0 if improved else score_wait+1
            nll_improved=np.isfinite(val_result['nll']) and val_result['nll']<best_nll-MIN_DELTA
            best_nll=val_result['nll'] if nll_improved else best_nll
            nll_wait=0 if nll_improved else nll_wait+1
        finally:
            ema.restore(net)
        scheduler.step()
        print(f'MLP seed {seed} Epoch {epoch:03d}/{EPOCHS} | Val EMD={val_result["emd"]:.5f} MAE={val_result["mae"]:.4f} NLL={val_result["nll"]:.4f} Spearman={val_result["spearman"]:.4f} | wait={score_wait}/{PATIENCE}',flush=True)
        if score_wait>=PATIENCE or nll_wait>=NLL_PATIENCE:
            break
    if best_state is None or best_val is None:
        raise RuntimeError(f'MLP seed {seed} 没有得到有效验证结果')
    net.load_state_dict(best_state,strict=True)
    test_probabilities=predict_feature_distribution(net,test_feature_values)
    test_metrics=compute_metrics(test_probabilities.numpy(),test_targets.numpy())
    result={'seed':seed,'best_epoch':best_epoch,'stop_epoch':epoch,'trainable_params':sum(param.numel() for param in net.parameters() if param.requires_grad)}
    result.update({f'val_{key}':float(value) for key,value in best_val.items() if key in ('emd','mae','nll','spearman')})
    result.update({f'test_{key}':float(value) for key,value in test_metrics.items() if key in ('emd','mae','nll','spearman')})
    del net,optimizer,scheduler,scaler,ema,best_state,test_probabilities
    gc.collect()
    if device.type=='cuda':
        torch.cuda.empty_cache()
    return result
mlp_seed_rows=[train_mlp_seed(seed) for seed in SEEDS]
mlp_frame=pd.DataFrame.from_records(mlp_seed_rows)
main_table=pd.read_csv(ROOT_DIR/'outputs'/'tables'/'main_comparison.csv')
full_row=main_table.loc[main_table['model'].astype(str).str.lower().eq('ours')].iloc[0]
summary_rows=[]
for method,frame in [('Full SSE',None),('Parameter-matched MLP',mlp_frame)]:
    if method=='Full SSE':
        row={'model':method,'trainable_params':PARAMETER_COUNTS[method],'seed_count':5}
        for metric in ('emd','mae','nll','spearman'):
            row[f'{metric}_mean']=float(full_row[f'{metric}_mean'])
            row[f'{metric}_sd']=float(full_row[f'{metric}_seed_sd'])
    else:
        row={'model':method,'trainable_params':int(frame['trainable_params'].iloc[0]),'seed_count':len(frame)}
        for metric in ('emd','mae','nll','spearman'):
            row[f'{metric}_mean']=float(frame[f'test_{metric}'].mean())
            row[f'{metric}_sd']=float(frame[f'test_{metric}'].std(ddof=1))
    summary_rows.append(row)
summary_frame=pd.DataFrame.from_records(summary_rows)
output_frame=pd.concat([summary_frame.assign(row_type='summary'),mlp_frame.assign(model='Parameter-matched MLP',row_type='seed')],ignore_index=True,sort=False)
output_path=TABLE_DIR/'parameter_matched_mlp.csv'
output_frame.to_csv(output_path,index=False,encoding='utf-8-sig',float_format='%.8f')
print(summary_frame.to_string(index=False,float_format=lambda value:f'{value:.6f}'))
print(f'参数量匹配对照结果已保存: {output_path}')

device: cuda PyTorch: 2.10.0+cu128
method: sparse score-field equilibrium head on frozen TIPSV2 features


训练图像: 2335 验证图像: 260
训练集单评分照片: 969
训练集原始先验: [0.027899999171495438, 0.05999999865889549, 0.15459999442100525, 0.22040000557899475, 0.19629999995231628, 0.1655000001192093, 0.10249999910593033, 0.041099999099969864, 0.021800000220537186, 0.00989999994635582]


Epoch 01/200 | Train EMD=0.30501 NLL=2.2709 | Val EMD=0.30564 MAE=1.5515 NLL=2.2958 Spearman=0.3230 | wait=0/40 nll_wait=0/6


Epoch 02/200 | Train EMD=0.27785 NLL=2.1158 | Val EMD=0.30444 MAE=1.5417 NLL=2.2884 Spearman=0.3654 | wait=0/40 nll_wait=0/6


Epoch 03/200 | Train EMD=0.24297 NLL=1.9044 | Val EMD=0.30164 MAE=1.5197 NLL=2.2707 Spearman=0.4139 | wait=0/40 nll_wait=0/6


Epoch 04/200 | Train EMD=0.22873 NLL=1.8241 | Val EMD=0.29717 MAE=1.4860 NLL=2.2424 Spearman=0.4397 | wait=0/40 nll_wait=0/6


... (183 outputs left out)

MLP seed 777 Epoch 030/200 | Val EMD=0.21323 MAE=0.9920 NLL=1.7685 Spearman=0.5285 | wait=3/40


MLP seed 777 Epoch 031/200 | Val EMD=0.21299 MAE=0.9911 NLL=1.7716 Spearman=0.5263 | wait=4/40


MLP seed 777 Epoch 032/200 | Val EMD=0.21281 MAE=0.9904 NLL=1.7748 Spearman=0.5258 | wait=5/40


                model  trainable_params  seed_count  emd_mean   emd_sd  mae_mean   mae_sd  nll_mean   nll_sd  spearman_mean  spearman_sd
             Full SSE             72185           5  0.220432 0.006001  1.045211 0.027604  1.785488 0.033777       0.644142     0.016995
Parameter-matched MLP             72189           5  0.210326 0.003981  0.974892 0.020576  1.705791 0.022708       0.663526     0.005144
参数量匹配对照结果已保存: D:\MyJupyter\Works\FaceScore\outputs\tables\parameter_matched_mlp.csv
